# Bengaluru GEE extraction: LST, NDVI, NDBI, ALBEDO

This notebook authenticates to Google Earth Engine, computes Land Surface Temperature (LST) from Landsat 8 L2, computes NDVI / NDBI / ALBEDO from Sentinel-2, samples pixels and exports a CSV with columns: LST, NDVI, NDBI, ALBEDO, latitude, longitude.

Configuration is read from environment variables. Ensure you set GOOGLE_APPLICATION_CREDENTIALS and GEE_SERVICE_ACCOUNT for service-account auth, or run `earthengine authenticate` for interactive auth.

In [36]:
!pip install -q earthengine-api geemap pandas

In [38]:
import ee

PROJECT_ID = "isro-hackathon-499708"

try:
    ee.Initialize(project=PROJECT_ID)
except:
    ee.Authenticate()
    ee.Initialize(project=PROJECT_ID)

print("Earth Engine initialized")

Earth Engine initialized


In [39]:
!pip install -q geemap

import geemap

In [40]:
AOI = ee.Geometry.Rectangle(
    [77.45, 12.80, 77.80, 13.20]
)

Map = geemap.Map()
Map.centerObject(AOI, 10)
Map.addLayer(AOI, {}, "AOI")
Map

Map(center=[13.000004851285539, 77.62500000000011], controls=(WidgetControl(options=['position', 'transparent_…

In [41]:
START_DATE = "2024-01-01"
END_DATE = "2024-12-31"

In [42]:
def get_lst():

    col = (
        ee.ImageCollection("LANDSAT/LC08/C02/T1_L2")
        .filterBounds(AOI)
        .filterDate(START_DATE, END_DATE)
    )

    def convert(img):

        lst = (
            img.select("ST_B10")
            .multiply(0.00341802)
            .add(149.0)
            .subtract(273.15)
            .rename("LST")
        )

        return lst

    return col.map(convert).median().clip(AOI)

LST_img = get_lst()

print(
    LST_img.reduceRegion(
        reducer=ee.Reducer.minMax(),
        geometry=AOI,
        scale=1000,
        maxPixels=1e9
    ).getInfo()
)

{'LST_max': 35.22030953000001, 'LST_min': 22.888093370000007}


In [43]:
s2 = (
    ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
    .filterBounds(AOI)
    .filterDate(START_DATE, END_DATE)
    .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 20))
    .select(["B2","B3","B4","B8","B11"])
)

s2_med = s2.median().clip(AOI)

# CRITICAL
s2_med = s2_med.divide(10000)

In [44]:
NDVI_img = (
    s2_med.normalizedDifference(
        ["B8","B4"]
    )
    .rename("NDVI")
)

print(
    NDVI_img.reduceRegion(
        ee.Reducer.minMax(),
        AOI,
        1000,
        maxPixels=1e9
    ).getInfo()
)

{'NDVI_max': 0.7580340264650283, 'NDVI_min': -0.22816399286987527}


In [45]:
NDBI_img = (
    s2_med.normalizedDifference(
        ["B11","B8"]
    )
    .rename("NDBI")
)

print(
    NDBI_img.reduceRegion(
        ee.Reducer.minMax(),
        AOI,
        1000,
        maxPixels=1e9
    ).getInfo()
)

{'NDBI_max': 0.22654491672561247, 'NDBI_min': -0.41768292682926833}


In [51]:
ALBEDO_img = (
    s2_med.select("B2")
    .add(s2_med.select("B3"))
    .add(s2_med.select("B4"))
    .add(s2_med.select("B8"))
    .divide(4)
    .rename("ALBEDO")
)

print(
    ALBEDO_img.reduceRegion(
        ee.Reducer.minMax(),
        AOI,
        1000,
        maxPixels=1e9
    ).getInfo()
)


{'ALBEDO_max': 0.24417500000000003, 'ALBEDO_min': 0.0352125}


In [56]:
EVI = s2_med.expression(
    '2.5*((NIR-RED)/(NIR+6*RED-7.5*BLUE+1))',
    {
        'NIR': s2_med.select('B8'),
        'RED': s2_med.select('B4'),
        'BLUE': s2_med.select('B2')
    }
).rename('EVI')

In [57]:
MNDWI = s2_med.normalizedDifference(
    ['B3','B11']
).rename('MNDWI')

In [59]:
stack = (
    LST_img
    .addBands(NDVI_img)
    .addBands(NDBI_img)
    .addBands(ALBEDO_img)
    .addBands(EVI)
    .addBands(MNDWI)
)

In [60]:
sample_fc = stack.sample(
    region=AOI,
    scale=30,
    numPixels=1000,
    geometries=True
)

print(sample_fc.size().getInfo())

1000


In [61]:
task = ee.batch.Export.table.toDrive(
    collection=sample_fc,
    description="urban_heat_dataset",
    fileFormat="CSV"
)

task.start()

print("Export started")

Export started


In [62]:
ee.batch.Task.list()

[<Task MFVE442IJRVZS7EKQXR735FB EXPORT_FEATURES: urban_heat_dataset (READY)>,
 <Task D66NWW42MEA7WHT3LPPRRLEA EXPORT_FEATURES: urban_heat_dataset (COMPLETED)>,
 <Task ZPTQ7AGXIM5H4SWJT75SILMS EXPORT_FEATURES: urban_heat_dataset (COMPLETED)>]

In [63]:
print(sample_fc.first().getInfo()["properties"])

{'ALBEDO': 0.204325, 'EVI': 0.15601963471111652, 'LST': 27.09225797000005, 'MNDWI': -0.28240274138278576, 'NDBI': 0.04073286438737114, 'NDVI': 0.1811864236076141}


In [64]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive
